In [ ]:
import pandas as pd
import joblib
from pathlib import Path


X_test_scaled = pd.read_csv(
    "processed/X_test_scaled.csv"
)

y_test = pd.read_csv(
    "processed/y_test.csv"
).squeeze()

print("Test data:", X_test_scaled.shape)
print("Test labels:", y_test.shape)

print(y_test.value_counts())

Test data: (56962, 30)
Test labels: (56962,)
Class
0    56864
1       98
Name: count, dtype: int64


In [2]:
baseline_models = {
    "Logistic Regression": joblib.load(
        "models/logistic_regression.pkl"
    ),

    "Decision Tree": joblib.load(
        "models/decision_tree.pkl"
    ),

    "Random Forest": joblib.load(
        "models/random_forest.pkl"
    ),

    "KNN": joblib.load(
        "models/k_nearest_neighbors.pkl"
    ),

    "SVM": joblib.load(
        "models/support_vector_machine.pkl"
    ),

    "Naive Bayes": joblib.load(
        "models/naive_bayes.pkl"
    )
}

In [3]:
tuned_models = {
    "Logistic Regression": joblib.load(
        "models/tuned_logistic_regression.pkl"
    ),

    "Decision Tree": joblib.load(
        "models/tuned_decision_tree.pkl"
    ),

    "Random Forest": joblib.load(
        "models/tuned_random_forest.pkl"
    ),

    "KNN": joblib.load(
        "models/tuned_knn.pkl"
    ),

    "SVM": joblib.load(
        "models/tuned_svm.pkl"
    ),

    "Naive Bayes": joblib.load(
        "models/tuned_naive_bayes.pkl"
    )
}

In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

In [15]:
def evaluate_models(models, X_test, y_test):

    results = []

    for name, model in models.items():

        
        y_pred = model.predict(X_test)

        
        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(X_test)[:, 1]

        else:
            y_score = model.decision_function(X_test)

      
        accuracy = accuracy_score(y_test, y_pred)

        precision = precision_score(
            y_test,
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_test,
            y_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            y_pred,
            zero_division=0
        )

        roc_auc = roc_auc_score(
            y_test,
            y_score
        )
        pr_auc = average_precision_score(
            y_test,
            y_score
        )
        results.append({
            "Model": name,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "ROC-AUC": roc_auc,
            "Precision-Recall":pr_auc
            
        })

    return pd.DataFrame(results)

In [16]:
baseline_results = evaluate_models(
    baseline_models,
    X_test_scaled,
    y_test
)

baseline_results

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Precision-Recall
0,Logistic Regression,0.960219,0.038330,0.918367,0.073590,0.976126,0.656263
1,Decision Tree,0.903286,0.015924,0.908163,0.031299,0.905721,0.014620
2,Random Forest,0.964116,0.042333,0.918367,0.080935,0.977697,0.695286
3,KNN,0.972701,0.053889,0.897959,0.101675,0.963197,0.200196
4,SVM,0.981058,0.075325,0.887755,0.138867,0.979027,0.599726
5,Naive Bayes,0.960869,0.037343,0.877551,0.071637,0.958773,0.054649


In [17]:
tuned_results = evaluate_models(
    tuned_models,
    X_test_scaled,
    y_test
)

tuned_results

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Precision-Recall
0,Logistic Regression,0.986675,0.104192,0.887755,0.186495,0.978458,0.670483
1,Decision Tree,0.908290,0.016598,0.897959,0.032593,0.931234,0.050592
2,Random Forest,0.968681,0.047747,0.908163,0.090724,0.975493,0.693777
3,KNN,0.987658,0.112676,0.897959,0.200228,0.969433,0.468434
4,SVM,0.981058,0.075325,0.887755,0.138867,0.979027,0.599726
5,Naive Bayes,0.960869,0.037343,0.877551,0.071637,0.958773,0.054649


In [8]:
baseline_results["Type"] = "Baseline"
tuned_results["Type"] = "Tuned"

In [9]:
comparison = pd.concat(
    [
        baseline_results,
        tuned_results
    ],
    ignore_index=True
)

comparison

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Type
0,Logistic Regression,0.960219,0.038330,0.918367,0.073590,0.976126,Baseline
1,Decision Tree,0.903286,0.015924,0.908163,0.031299,0.905721,Baseline
2,Random Forest,0.964116,0.042333,0.918367,0.080935,0.977697,Baseline
3,KNN,0.972701,0.053889,0.897959,0.101675,0.963197,Baseline
4,SVM,0.981058,0.075325,0.887755,0.138867,0.979027,Baseline
5,Naive Bayes,0.960869,0.037343,0.877551,0.071637,0.958773,Baseline
6,Logistic Regression,0.986675,0.104192,0.887755,0.186495,0.978458,Tuned
7,Decision Tree,0.908290,0.016598,0.897959,0.032593,0.931234,Tuned
8,Random Forest,0.968681,0.047747,0.908163,0.090724,0.975493,Tuned
9,KNN,0.987658,0.112676,0.897959,0.200228,0.969433,Tuned
